In [1]:
import torch
import torch.nn as nn

In [3]:
# 1. Data Preparation
text = "I love machine learning and I love deep learning"
words = text.split()
# Create vocabulary
vocab = sorted(set(words))
word_to_ix = {word: i for i, word in enumerate(vocab)}
ix_to_word = {i: word for word, i in word_to_ix.items()}
print("Vocabulary:", vocab)
# 2. Generate Training Data
# Use the previous and next words as context
data = []
for i in range(1, len(words) - 1):
    context = [words[i - 1], words[i + 1]]
    target = words[i]
    data.append((context, target))
print("\nTraining data:")
for context, target in data:
    print(f"Context: {context} -> Target: {target}")
# 3. Define the CBOW Model
class CBOW(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.linear = nn.Linear(embedding_dim, vocab_size)

    def forward(self, context):
        embeds = self.embeddings(context)
        embeds = embeds.mean(dim=0)
        output = self.linear(embeds)
        return output
# Initialize model
torch.manual_seed(42)
model = CBOW(len(vocab), 10)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
#4. Train the Model
for epoch in range(100):
    total_loss = 0
    for context, target in data:
        inputs = torch.tensor(
            [word_to_ix[word] for word in context],
            dtype=torch.long
        )
        label = torch.tensor(
            [word_to_ix[target]],
            dtype=torch.long
        )
        output = model(inputs)
        loss = loss_fn(output.unsqueeze(0), label)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 20 == 0:
        print(
            f"Epoch {epoch + 1}, "
            f"Loss: {total_loss:.4f}"
        )
# 5. Predict the Target Word
context = ["I", "learning"]

inputs = torch.tensor(
    [word_to_ix[word] for word in context],
    dtype=torch.long
)
model.eval()
with torch.no_grad():
    output = model(inputs)
    predicted_index = output.argmax().item()
    predicted_word = ix_to_word[predicted_index]
print("\nContext words:", context)
print("Predicted word:", predicted_word)

Vocabulary: ['I', 'and', 'deep', 'learning', 'love', 'machine']

Training data:
Context: ['I', 'machine'] -> Target: love
Context: ['love', 'learning'] -> Target: machine
Context: ['machine', 'and'] -> Target: learning
Context: ['learning', 'I'] -> Target: and
Context: ['and', 'love'] -> Target: I
Context: ['I', 'deep'] -> Target: love
Context: ['love', 'learning'] -> Target: deep
Epoch 20, Loss: 9.2621
Epoch 40, Loss: 7.0552
Epoch 60, Loss: 5.6766
Epoch 80, Loss: 4.7172
Epoch 100, Loss: 4.0308

Context words: ['I', 'learning']
Predicted word: and


In [ ]:
%pip install torch